# Prisma — Anexos (experimentos y diagnósticos)

**Nada de este cuaderno forma parte del flujo principal** (`Prisma_unificado.ipynb`). Reúne el Paso 17 y los
anexos A.1–A.17 que estaban en `Acredit_Calidad.ipynb`, como evidencia de las decisiones de diseño.

| Anexos | Estado | Qué necesitan |
|---|---|---|
| **A.1–A.15** | **Históricos.** Analizan una corrida antigua de Calidad/evaluación docente (`20260922-074634_622773ba80`). Se conservan como referencia y **no forman parte de la validación**. | La carpeta de esa corrida bajo `RAIZ/outputs/ft/`. Si todavía la tiene en Drive, cópiela a la carpeta de trabajo para reproducirlos; si no, quedan como documentación. |
| **A.16–A.17** | Vigentes: se pueden volver a ejecutar. Están **al final**. | **Requieren `resultado`** (y `cfg_final`, `encuestas`, `FOCAL`) de una corrida del pipeline unificado **en la misma sesión**: ejecute antes `Prisma_unificado.ipynb`. |

Las celdas de este cuaderno solo preparan el entorno (`RAIZ`, módulo, léxicos). La celda «FRENO» evita que
«Ejecutar todo» dispare los anexos por accidente.


In [ ]:
import sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    !pip install -q bertopic pysentimiento openpyxl pyarrow

# Carpeta del proyecto en Drive (solo Colab). Local: RAIZ se detecta por pyproject.toml.
RAIZ_COLAB = "/content/drive/MyDrive/capstone_nlp_mine009"


In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*n_jobs value.*")

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 9
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 180)

AZUL, NARANJA, VERDE, MORADO, ROJO, GRIS = (
    "#2c6fbb", "#c8811d", "#3d8b64", "#7a5195", "#c1453c", "#b8bcc2")
COLOR_SENT = {"POS": VERDE, "NEU": "#9aa0a6", "NEG": ROJO}

GUARDAR_EN_DRIVE = True  # persiste outputs y caché de embeddings entre sesiones


def raiz_proyecto(marcador="pyproject.toml"):
    for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (c / marcador).exists():
            return c
    raise FileNotFoundError(marcador)


if EN_COLAB and GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path(RAIZ_COLAB)
elif EN_COLAB:
    RAIZ = Path("/content")
else:
    RAIZ = raiz_proyecto()

for p in (RAIZ, RAIZ / "src", RAIZ / "src" / "datacleaning"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

BASE_SALIDA = RAIZ
SALIDA = BASE_SALIDA / "outputs" / "ft"
CACHE = BASE_SALIDA / "cache" / "embeddings_ft"
SALIDA.mkdir(parents=True, exist_ok=True)
print("Proyecto:", RAIZ, "| salidas:", SALIDA)

In [ ]:
import sys
from pathlib import Path

# Verifica que el módulo exista en RAIZ (o RAIZ/src) antes de importarlo.
module_name = "Mod_Bertopic_Ft.py"
possible_module_paths = [RAIZ / module_name, RAIZ / "src" / module_name]
module_found = False
for p in possible_module_paths:
    if p.exists():
        print(f"Módulo '{module_name}' encontrado en: {p}")
        module_found = True
        break

if not module_found:
    error_msg = (
        f"Error: El módulo '{module_name}' no se encontró en las rutas esperadas.\n"
        f"Por favor, asegúrese de que el archivo '{module_name}' esté en:\n"
        f"  - '{RAIZ}'\n"
        f"  - o en '{RAIZ / 'src'}'\n"
        "Si está utilizando Google Colab, asegúrese de que el archivo esté en la carpeta de su Google Drive a la que apunta 'RAIZ'."
    )
    raise FileNotFoundError(error_msg)

import Mod_Bertopic_Ft as ft

print("Versión del módulo:", ft.VERSION)
assert ft.VERSION.startswith("ft-r4"), "Suba el Mod_Bertopic_Ft.py con los cambios de equipo (ft-r4)"

import hashlib
_huella_disco = hashlib.md5(Path(ft.__file__).read_bytes()).hexdigest()
_huella_memoria = getattr(ft, "HUELLA_ARCHIVO", "(módulo anterior a esta verificación)")
print("Huella en memoria:", _huella_memoria)
print("Huella en disco:  ", _huella_disco)
assert _huella_memoria == _huella_disco, (
    "El módulo cargado NO es el que está en Drive. Entorno de ejecución -> Reiniciar sesión, "
    "y vuelva a correr desde el Paso 0.")

In [ ]:
recursos = ft.RecursosLexicos.cargar(RAIZ)  # léxicos fusionados: un solo juego en RAIZ


## Paso 17 — Qué traerme para la auditoría final

De ~~`outputs/ft/evaluacion_docente_p19/<run_id>/`~~ **[ft-r4.10]** `outputs/ft/<encuesta>_<clave_logica>/<run_id>/` (p. ej. `outputs/ft/evadoc_evaluacion_obs19_aspectos_resaltar/…`):

| Archivo | Para qué |
|---|---|
| `scorecard.json` | Puertas que fallan: definen qué se ajusta |
| `auditoria.xlsx` (diligenciado) | Pureza por tópico y F1: el único juicio que no se puede automatizar |
| `grid.csv` | Forma del espacio de búsqueda: dónde mover la rejilla |
| `topicos.csv` · `absa.csv` · ~~`absa_taxonomia.csv`~~ | Lectura editorial de aspectos, nombrado y prioridades. **[ft-r4.10]** `absa_taxonomia.csv` ya no se genera; `absa.csv` trae `confiable` y `rango_prioridad`, y `clausulas.csv` la columna `asignacion` |

Además, anote lo observado en las muestras del Paso 3 (falsos `NO_OPINION` o `VALORACION_GLOBAL`).

**Alcance de la ronda 2 (sólo refinamiento):** `prob_aspecto_min`, `margen_aspecto_min` y
`acuerdo_nombrado_min` del zero-shot, redacción de las descripciones de la taxonomía, umbrales de reasignación y fusión, rango de
`min_topic_size`, `margen_neu`, léxicos de neutralización, puertas de `CriteriosNegocio` con
justificación, y la rejilla `"completa"` para la corrida de entrega. No hay cambios de arquitectura.

> **[ft-r4.10]** El "alcance de la ronda 2" es histórico: el zero-shot (`prob_aspecto_min`, `margen_aspecto_min`, `acuerdo_nombrado_min`) se retiró en H16. Pendientes vigentes de la ronda 6: decisión sobre la G3 (H20), validar la regla `confiable` en obs20, términos `_PENDIENTES` de `neutralizacion.xlsx` (*nota*, *gente*), frases en `stopwords.xlsx` que el vectorizador ignora, y avisar al equipo de `neutralizacion.xlsx` obligatorio, `MIS_ENCUESTAS` y las nuevas claves de stopwords.

In [ ]:
# ⛔ FRENO — fin del flujo principal (Pasos 0–17).
# "Ejecutar todo" se detiene aquí a propósito: el Anexo son diagnósticos 🧪 (algunos tardan
# minutos o dependen de carpetas de prueba que ya no existen). Para correrlo, hágalo celda por
# celda, o cambie CORRER_ANEXO a True.
CORRER_ANEXO = False
if not CORRER_ANEXO:
    raise RuntimeError(
        "⛔ Fin del flujo principal. El Anexo es de diagnóstico: córralo celda por celda "
        "o cambie CORRER_ANEXO = True en esta celda."
    )

---

# Anexo — Diagnóstico T2 (ronda Cris, 2026-09-22)

**Nada de este anexo forma parte del flujo.** Son las pruebas 🧪 que justificaron el cambio `ft-r4.10` del ABSA. Se conservan como evidencia, con sus salidas.

**Cómo correrlo:** Paso 0 + Paso 1, luego el anexo en orden. Lee la corrida de referencia `20260922-074634_622773ba80` desde `outputs/`; no re-corre el pipeline. Las variables llevan prefijo `dx_` para no pisar las del flujo principal.

**Hallazgos en una línea cada uno:**
1. T2 (`pesar · tener · dificil`, 4.155 cláusulas) salía #1 en la priorización.
2. Solo 0,7% de T2 contiene "pesar": la etiqueta c-TF-IDF muestra palabras *distintivas*, no *frecuentes*.
3. 60% de T2 son textos de aspecto de **una palabra** (corpus: 18,5%). El encoder agrupa por forma: "clara" sola → T2; 613 frases con "clara" → T6.
4. 85% de las palabras sueltas las produce la neutralización, y casi siempre correctamente (quita persona, valorativos y relleno).
5. Plantilla en el encoder y voto de frases: no resuelven.
6. Regla final: **no confiable = cohesión baja (`revisar`) Y ≥ 2× palabras sueltas del corpus.** Solo T2 (3,25×). Cualquier umbral entre 1,8 y 3,2 da el mismo resultado.

**Pendiente:** validar la regla en obs20.

> ⚠️ **A.1–A.15: HISTÓRICOS.** Dependen de una corrida concreta (`20260922-074634_622773ba80`) y se
> conservan como referencia; **no forman parte de la validación**. Para reproducirlos, copie esa
> carpeta de corrida (si aún está en Drive) bajo `RAIZ/outputs/ft/`; si no, son documentación.


### A.1 — Cargar la corrida de referencia (sin re-correr el pipeline)

Requiere haber corrido solo el Paso 0 y el Paso 1. No usa `resultado`.

In [ ]:
# 🧪 Anexo A.1 — Cargar la corrida de referencia (sin re-correr el pipeline)
DX_RAIZ = RAIZ
dx_encontradas = sorted(DX_RAIZ.rglob("20260922-074634_622773ba80"))
print(dx_encontradas)                      # debe mostrar UNA ruta, sin _V1

DX_CARPETA = dx_encontradas[0]
dx_cl = pd.read_parquet(next(q for q in (DX_CARPETA / "clausulas_detalle.parquet", DX_CARPETA / "clausulas.parquet") if q.exists()))
dx_topicos = pd.read_csv(DX_CARPETA / "topicos.csv")
print(dx_cl.shape)                         # debe dar (40871, 20)
print(dx_topicos.loc[dx_topicos["topico"] == 2, ["etiqueta", "clausulas"]])   # T2 con 4155

### A.2 — ¿Qué hay dentro de T2? (la etiqueta engaña)

In [ ]:
# 🧪 Anexo A.2 — ¿Qué hay dentro de T2? (la etiqueta engaña)
dx_t2 = dx_cl[dx_cl["topico"] == 2]
print(f"{len(dx_t2):,} cláusulas en T2\n")

# A) ¿Qué parte de T2 contiene "pesar"?
pct_pesar = dx_t2['texto_aspecto'].str.contains(r'\bpesar\b').mean()
print(f"con 'pesar': {pct_pesar:.1%}\n")

# B) ¿Cuántas palabras le quedaron a cada cláusula después de neutralizar?
print("palabras en el texto de aspecto:")
print(dx_t2["texto_aspecto"].str.split().str.len().value_counts().sort_index().head(6), "\n")

# C) Los 25 textos de aspecto más repetidos
print(dx_t2["texto_aspecto"].value_counts().head(25))

### A.3 — ¿Dónde caen las cláusulas de UNA palabra?

In [ ]:
# 🧪 Anexo A.3 — ¿Dónde caen las cláusulas de UNA palabra?
dx_asp = dx_cl[dx_cl["motivo"] == "ASPECTO"].copy()
dx_asp["n_pal"] = dx_asp["texto_aspecto"].str.split().str.len()
dx_una = dx_asp[dx_asp["n_pal"] == 1]

print(f"cláusulas de aspecto con 1 palabra: {len(dx_una):,} ({len(dx_una)/len(dx_asp):.1%} del total)")
print(f"de esas, cuántas cayeron en T2: {(dx_una['topico'] == 2).mean():.1%}\n")
print("tópicos donde cayeron las de 1 palabra:")
print(dx_una["topico"].value_counts().head(8))

### A.4 — Misma palabra, distinto tópico según el largo del texto

In [ ]:
# 🧪 Anexo A.4 — Misma palabra, distinto tópico según el largo del texto
dx_t2_una = dx_t2[dx_t2["texto_aspecto"].str.split().str.len() == 1]
palabras = dx_t2_una["texto_aspecto"].value_counts().head(10).index
dx_multi = dx_asp[dx_asp["n_pal"] > 1]

for p in palabras:
    con_p = dx_multi[dx_multi["texto_aspecto"].str.contains(rf"\b{p}\b")]
    destinos = con_p["topico"].value_counts().head(3).to_dict()
    print(f"{p:14s} sola -> T2 | en frases ({len(con_p):4d}) -> {destinos}")

### A.5 — Prueba: plantilla fija en el encoder (no concluyente)

Tarda unos minutos (carga el encoder). Resultado: la plantilla acerca las palabras a su tópico pero casi nunca las voltea; además un prefijo común comprime todos los márgenes hacia 0. No concluyente → no se implementó.

In [ ]:
# 🧪 Anexo A.5 — Prueba: plantilla fija en el encoder (no concluyente)
from sentence_transformers import SentenceTransformer
enc = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

def vec(textos, plantilla):
    return enc.encode([plantilla.format(t) for t in textos],
                      normalize_embeddings=True, show_progress_bar=False)

def centroide(textos, plantilla):
    v = vec(textos, plantilla).mean(axis=0)
    return v / np.linalg.norm(v)

PLANTILLAS = {"1_sin": "{}",
              "2_aspecto": "Aspecto: {}",
              "3_docente": "El docente se destaca por: {}"}

filas = []
for p in palabras:
    patron = rf"\b{p}\b"
    frases_p = dx_multi[dx_multi["texto_aspecto"].str.contains(patron)]
    dest = frases_p.loc[~frases_p["topico"].isin([-1, 2]), "topico"].value_counts().idxmax()
    # centroides SIN la palabra evaluada, para no hacer trampa
    pool_dest = dx_asp.loc[(dx_asp["topico"] == dest) & ~dx_asp["texto_aspecto"].str.contains(patron), "texto_aspecto"].drop_duplicates()
    pool_t2 = dx_t2.loc[~dx_t2["texto_aspecto"].str.contains(patron), "texto_aspecto"].drop_duplicates()
    m_dest = pool_dest.sample(min(100, len(pool_dest)), random_state=0).tolist()
    m_t2 = pool_t2.sample(min(100, len(pool_t2)), random_state=0).tolist()
    for nombre, pl in PLANTILLAS.items():
        v = vec([p], pl)[0]
        margen = float(v @ centroide(m_dest, pl)) - float(v @ centroide(m_t2, pl))
        filas.append({"palabra": p, "destino": dest, "plantilla": nombre, "margen": round(margen, 3)})

res = pd.DataFrame(filas).pivot(index=["palabra", "destino"], columns="plantilla", values="margen")
display(res)
print("\npalabras que quedan más cerca de su destino que de T2:")
print((res > 0).sum())

### A.6 — Prueba: reasignar por voto de frases (descartada)

Rescata 98 de 2.496 palabras sueltas de T2 y la mitad irían a T0 (otro cajón de sastre). Descartada.

In [ ]:
# 🧪 Anexo A.6 — Prueba: reasignar por voto de frases (descartada)
dx_multi_ok = dx_multi[~dx_multi["topico"].isin([-1, 2])]   # solo frases con tópico "confiable"
pares = (dx_multi_ok.assign(palabra=dx_multi_ok["texto_aspecto"].str.split())
                 .explode("palabra")[["palabra", "topico"]])

votos = pares.groupby("palabra")["topico"].agg(
    n="size",
    destino=lambda s: s.value_counts().idxmax(),
    share=lambda s: s.value_counts(normalize=True).iloc[0],
)

dx_una_v = dx_una.merge(votos, left_on="texto_aspecto", right_index=True, how="left")
ok = (dx_una_v["n"] >= 20) & (dx_una_v["share"] >= 0.5)

print(f"cláusulas de 1 palabra: {len(dx_una_v):,}")
print(f"reasignables con confianza: {ok.sum():,} ({ok.mean():.1%})")
print(f"  de las que hoy están en T2: {(ok & (dx_una_v['topico'] == 2)).sum():,} de {(dx_una_v['topico'] == 2).sum():,}")
print("\ndestino de las reasignables que hoy están en T2:")
print(dx_una_v.loc[ok & (dx_una_v["topico"] == 2), "destino"].value_counts().head(8))

### A.7 — Origen de las palabras sueltas (neutralización / respuesta corta / segmentador)

In [ ]:
# 🧪 Anexo A.7 — Origen de las palabras sueltas (neutralización / respuesta corta / segmentador)
dx_una = dx_una.copy()

# ¿Cuántas palabras tenía la cláusula ORIGINAL, antes de neutralizar?
dx_una["pal_original"] = dx_una["clausula"].str.split().str.len()

# ¿Cuántas cláusulas tenía la respuesta completa del estudiante?
dx_una["clausulas_doc"] = dx_una["doc_id"].map(dx_cl.groupby("doc_id").size())

dx_una["origen"] = "B. neutralización recortó"
dx_una.loc[dx_una["pal_original"] <= 2, "origen"] = "C. fragmento corto"
dx_una.loc[(dx_una["pal_original"] <= 2) & (dx_una["clausulas_doc"] == 1), "origen"] = "A. respuesta de 1-2 palabras"

print(dx_una["origen"].value_counts(), "\n")
print("solo las que están en T2:")
print(dx_una.loc[dx_una["topico"] == 2, "origen"].value_counts(), "\n")
display(dx_una.sample(15, random_state=1)[["clausula", "texto_aspecto", "clausulas_doc", "origen"]])

### A.8 — ¿Qué retiró la neutralización? (relleno vs. información)

In [ ]:
# 🧪 Anexo A.8 — ¿Qué retiró la neutralización? (relleno vs. información)
from collections import Counter

dx_b_t2 = dx_una[dx_una["origen"].str.startswith("B") & (dx_una["topico"] == 2)]
display(dx_b_t2.sample(20, random_state=2)[["clausula", "texto_aspecto"]])

# Palabras que la neutralización retiró (sin contar las vacías tipo "de", "la")
quitadas = Counter()
for original, aspecto_ in zip(dx_b_t2["clausula_clean"], dx_b_t2["texto_aspecto"]):
    quedan = set(aspecto_.split())
    quitadas.update(w for w in original.split()
                    if w not in quedan and w not in recursos.stopwords_nltk)
print(quitadas.most_common(30))

### A.9 — ABSA v1: ordenar solo por `revisar` (DESCARTADA)

⚠️ La salida guardada es la de la versión DESCARTADA (T10 y T12 desaparecían del top). Si la re-ejecuta con el módulo actual verá el resultado final, igual al de A.12.

In [ ]:
# 🧪 Anexo A.9 — ABSA v1: ordenar solo por `revisar` (DESCARTADA)

dx_absa = ft.construir_absa(dx_cl, dx_topicos, ft.Config())
cols = ["etiqueta", "total_clausulas", "pct_negativo", "prioridad", "confiable", "rango_prioridad"]
display(dx_absa[dx_absa["rango_prioridad"].notna()][cols].head(10))

### A.10 — Por qué v1 falla: el umbral de cohesión es relativo

In [ ]:
# 🧪 Anexo A.10 — Por qué v1 falla: el umbral de cohesión es relativo
cols = ["etiqueta", "total_clausulas", "pct_negativo", "prioridad", "confiable", "rango_prioridad"]
print("Priorizables marcados como NO confiables:")
display(dx_absa[dx_absa["rango_prioridad"].notna() & ~dx_absa["confiable"]][cols])

print("\nLos 8 tópicos con menor cohesión:")
display(dx_topicos[["topico", "etiqueta", "cohesion", "revisar", "clausulas"]]
        .sort_values("cohesion").head(8))

### A.11 — Cohesión + palabras sueltas en los 28 tópicos

In [ ]:
# 🧪 Anexo A.11 — Cohesión + palabras sueltas en los 28 tópicos
dx_asp = dx_cl[dx_cl["motivo"] == "ASPECTO"].copy()
dx_asp["una"] = dx_asp["texto_aspecto"].str.split().str.len() == 1
base = dx_asp["una"].mean()

pct_una = dx_asp[dx_asp["topico"] >= 0].groupby("topico")["una"].mean().rename("pct_una")
t = dx_topicos.set_index("topico")[["etiqueta", "cohesion", "revisar", "clausulas"]].join(pct_una)
t["veces_base"] = (t["pct_una"] / base).round(2)

print(f"Palabras sueltas en todo el corpus: {base:.1%}\n")
display(t.sort_values("cohesion").round(3))

### A.12 — ABSA final (ft-r4.10): verificación

In [ ]:
# 🧪 Anexo A.12 — ABSA final (ft-r4.10): verificación
dx_absa = ft.construir_absa(dx_cl, dx_topicos, ft.Config())

cols = ["etiqueta", "total_clausulas", "pct_negativo", "prioridad",
        "veces_base_suelta", "confiable", "rango_prioridad"]
print("Top 6 de la priorización:")
display(dx_absa[dx_absa["rango_prioridad"].notna()][cols].head(6))
print("\nNo confiables:")
display(dx_absa[~dx_absa["confiable"]][cols])

---

## Punto 2 — Léxico de neutralización (ronda Cris, 2026-09-22)

**Qué se hizo:** (A) migrar las 233 palabras de las listas del código a `neutralizacion.xlsx` sin cambiar nada; (B) agregar 19 palabras nuevas clasificadas a mano.

**Criterio de clasificación (tres preguntas, en orden):**
1. ¿Encaja en algún código de la taxonomía de la encuesta? → es aspecto, no se toca.
2. ¿Tiene sentido *"mejore su ___"* / *"siga así con su ___"*? → es aspecto.
3. Si no: ¿dice **qué tan bueno** sin decir en qué? → VALORATIVO · ¿es **contexto o relleno**? → GENERICO · ¿habla de **quién es** la persona (identidad, apariencia)? → PERSONA.

Ante la duda, **no se retira**: quitar un aspecto real borra información en todo el corpus; dejar ruido solo ensucia un poco (y el punto 1 ya protege la priorización).

### A.13 — Lista de las 150 palabras sueltas más frecuentes para clasificar

Cubren 4.758 de 6.822 cláusulas de una palabra (70%). `pct_en_T2` sale siempre 0 o 100: cada texto único tiene un solo vector y cae en un solo tópico. Se clasificaron 15; las dudosas (*nota*, *gente*) quedaron en la hoja `_PENDIENTES` de `neutralizacion.xlsx`. Antes de compartir el Excel generado, borrar la columna `ejemplo` (texto de estudiantes).

In [ ]:
# 🧪 Anexo A.13 — Lista de palabras sueltas para clasificar
ref = sorted(RAIZ.rglob("20260922-074634_622773ba80"))[0]
dx_cl = pd.read_parquet(next(q for q in (ref / "clausulas_detalle.parquet", ref / "clausulas.parquet") if q.exists()))
dx_asp13 = dx_cl[dx_cl["motivo"] == "ASPECTO"]
dx_una13 = dx_asp13[dx_asp13["texto_aspecto"].str.split().str.len() == 1]

conteo = (dx_una13.groupby("texto_aspecto")
              .agg(n=("clausula", "size"),
                   pct_en_T2=("topico", lambda s: round(100 * (s == 2).mean())),
                   ejemplo=("clausula", "first"))
              .sort_values("n", ascending=False).head(150).reset_index())
conteo["CATEGORIA"] = ""
cubre = conteo["n"].sum()
print(f"Las 150 palabras cubren {cubre:,} de {len(dx_una13):,} cláusulas de una palabra ({cubre/len(dx_una13):.0%})")
conteo.to_excel(RAIZ / "clasificar_palabras_sueltas.xlsx", index=False)

### A.14 — Verificación de la migración y efecto de las palabras nuevas

⚠️ **Histórica:** compara el módulo SIN `neutralizacion.xlsx` (en `RAIZ`) contra el nuevo (en `RAIZ/_prueba_r410`). Solo tiene sentido antes de integrar; se conserva para reproducibilidad.

**Resultado (2026-09-22):**
- Migración: 0 términos perdidos en las tres categorías; agregados exactamente los 19 aprobados (8 valorativos, 2 persona, 9 genéricos). Verificado con el `accents.xlsx` real.
- Control: el módulo anterior reproduce el texto de aspecto guardado en **100,00%** de 36.957 cláusulas (el recálculo es fiel).
- Efecto: cambian 2.042 cláusulas; **257 pasan a valoración global**, 196 de ellas salían de T2 (4,7% de T2). El resto son retiros dentro de frases (*"verdad me ha ayudado"* → *"ayudado"*).
- Hallazgo: *nota* tiene al menos tres sentidos (jerga, calificación, *"se nota"*): confirma dejarla pendiente.

In [ ]:
# 🧪 Anexo A.14 — Verificar migración (A) y medir efecto (B)  [HISTÓRICA, ver nota]
import importlib.util, shutil, sys

PRUEBA = RAIZ / "_prueba_r410"
for patron in ["*negacion*.xlsx", "*stopword*.xlsx", "*accent*.xlsx", "*alerta*.xlsx"]:
    for f in RAIZ.glob(patron):
        shutil.copy(f, PRUEBA / f.name)

spec = importlib.util.spec_from_file_location("ft_nuevo", PRUEBA / "Mod_Bertopic_Ft.py")
ft_nuevo = importlib.util.module_from_spec(spec)
sys.modules["ft_nuevo"] = ft_nuevo
spec.loader.exec_module(ft_nuevo)

rec_v = ft.RecursosLexicos.cargar(RAIZ)
rec_n = ft_nuevo.RecursosLexicos.cargar(PRUEBA)
for cat in ["valorativos", "persona", "genericos"]:
    v, n = getattr(rec_v, cat), getattr(rec_n, cat)
    print(f"{cat:12s} viejo {len(v):3d} | nuevo {len(n):3d} | perdidas {sorted(v - n)} | agregadas {sorted(n - v)}")

ref = sorted(RAIZ.rglob("20260922-074634_622773ba80"))[0]
dx_cl = pd.read_parquet(next(q for q in (ref / "clausulas_detalle.parquet", ref / "clausulas.parquet") if q.exists()))
dx_asp14 = dx_cl[dx_cl["motivo"] == "ASPECTO"].copy()
orig, clean = dx_asp14["clausula"].fillna(""), dx_asp14["clausula_clean"].fillna("")
neu_v = ft.NeutralizadorAspecto(ft.Config(), rec_v)
neu_n = ft_nuevo.NeutralizadorAspecto(ft_nuevo.Config(), rec_n)
dx_asp14["recalc_viejo"] = [neu_v.neutralizar(o, c)[0] for o, c in zip(orig, clean)]
dx_asp14["recalc_nuevo"] = [neu_n.neutralizar(o, c)[0] for o, c in zip(orig, clean)]
print(f"Control: reproduce el texto guardado en {(dx_asp14['recalc_viejo'] == dx_asp14['texto_aspecto']).mean():.2%}")
cambian = dx_asp14["recalc_nuevo"] != dx_asp14["texto_aspecto"]
vacias = dx_asp14["recalc_nuevo"] == ""
en_t2 = dx_asp14["topico"] == 2
print(f"Cambian {cambian.sum():,}; pasan a valoración global {vacias.sum():,}; de T2: {(vacias & en_t2).sum():,} de {en_t2.sum():,}")

---

## Bloque 3 — Verificaciones (ronda Cris, 2026-09-22)

### A.15 — ¿El CSV de obs19 se leyó bien? (WARNING de separador, línea 193)

**Resultado:** el encabezado usa `,` (no `;`); 24.380 filas, 8 columnas, **0 columnas `Unnamed`**; `Periodo` (`9999-9`) e `ID de respuesta` (`999999999`) con formato consistente en el **100%** de las filas. `Código Cruce Autoevaluación` varía por diseño (códigos de curso). Además, el motor `python` de pandas falla si una fila trae un número distinto de campos, así que no hay filas corridas. **Conclusión:** falsa alarma; el `;` de una respuesta en la línea 193 rompía la lectura con `;`. Corregido en `ft-r4.10` (separador desde el encabezado).

In [ ]:
# 🧪 Anexo A.15 — ¿El CSV de obs19 se leyó bien?
import csv
dx_ruta = next(RAIZ.glob("evadoc_evaluacion__obs19*.csv"))
dx_encabezado = dx_ruta.read_text(encoding="utf-8", errors="replace").splitlines()[0]
print("Separador real del encabezado:", repr(csv.Sniffer().sniff(dx_encabezado).delimiter))

dx_marco = ft._leer_csv_institucion(dx_ruta)
print(f"Filas: {len(dx_marco):,} | columnas: {len(dx_marco.columns)}")
print("Columnas 'Unnamed' (señal de filas corridas):",
      [c for c in dx_marco.columns if str(c).startswith("Unnamed")])
for col in ["Periodo", "ID de respuesta", "Código Cruce Autoevaluación"]:
    if col in dx_marco.columns:
        formato = dx_marco[col].dropna().astype(str).str.replace(r"\d", "9", regex=True)
        dominante = formato.value_counts()
        print(f"{col:30s} vacíos: {dx_marco[col].isna().sum():5d} | formato dominante "
              f"'{dominante.index[0]}' en {dominante.iloc[0]:,} filas | otro formato: {len(formato) - dominante.iloc[0]:,}")

---

## Bloque 4 — Puertas que fallaron en la corrida 20260923-050625 (nota 5,9)

La nota es el % del peso de las puertas evaluadas que pasan (5,0 / 8,5 → 5,9). Fallaron G2 (ARI 0,608), G7 (redundancia 0,905), G8 (Spearman 0,21) y G9 (AUC 0,635). G2 → H23 (grid estable); G7 → H24 (fusión iterativa). **Nota:** la corrida 20260922-074634 NO es comparable: se hizo desde la carpeta de Julián con otro segmentador (mismo corpus: el segmentador actual reproduce la corrida nueva en 200/200 documentos cambiados y la vieja en 0/200) y sin calibrador.

> ⚠️ **A.16–A.17: requieren `resultado`.** Use `resultado`, `cfg_final`, `encuestas` y `FOCAL` de una
> corrida del pipeline unificado: ejecute `Prisma_unificado.ipynb` (hasta el Paso 7 como mínimo) en la
> misma sesión antes de correr estas celdas.


### A.16 — G7: pares de tópicos más parecidos, con y sin reasignadas

**Resultado:** el único par sobre el límite fue T0 (*aprendizaje · enseñar*) ↔ T4 (*aprendizaje · aula · dinámico*): 0,905 final y **0,896 sin reasignadas**, es decir, ya superaba el umbral de fusión (0,85) y no se fusionó porque la fusión era de una sola pasada. T4 resultó híbrido (texto más frecuente 7/1.036; 31% dinamismo → T3, 18% aprendizaje → T0, ~11% ambiente, demasiado pequeño para ser tópico). Decisión: fusión iterativa (H24) y `aula` → GENERICO. Costo aceptado: las ~110 cláusulas de *ambiente de aprendizaje* se diluyen. Requiere `resultado` en memoria.

In [ ]:
# 🧪 Anexo A.16 — G7: pares de tópicos más parecidos, antes y después de reasignar outliers
dx_mod = resultado.modelador
dx_emb, dx_final, dx_crudas = dx_mod.embeddings, dx_mod.etiquetas, dx_mod.etiquetas_hdbscan
dx_etiq = resultado.topicos.set_index("topico")["etiqueta"]

def dx_similitudes(etiquetas):
    cents = ft.centroides(dx_emb, etiquetas)
    ids = sorted(cents)
    M = np.stack([cents[i] for i in ids])
    S = M @ M.T
    return {(ids[a], ids[b]): float(S[a, b]) for a in range(len(ids)) for b in range(a + 1, len(ids))}

dx_s_final = dx_similitudes(dx_final)
dx_s_directa = dx_similitudes(np.where(dx_crudas != -1, dx_final, -1))
print("coseno final | sin reasignadas | par")
for (a, b), v in sorted(dx_s_final.items(), key=lambda x: -x[1])[:6]:
    print(f"   {v:.3f}     |     {dx_s_directa.get((a, b), float('nan')):.3f}      | {dx_etiq.get(a, a)}  <->  {dx_etiq.get(b, b)}")

### A.17 — G9: ¿el calibrador de sentimiento ayuda en obs19?

Misma corrida, mismas cláusulas; solo cambia calibrado vs. sin calibrar (las probabilidades originales de RoBERTuito quedan guardadas).

| versión | Spearman doc (G8) | AUC (G9) | monotonía | %NEG | %NEU |
|---|---|---|---|---|---|
| calibrado | 0,2115 | 0,6350 | 0,9636 | 4,4 | 3,5 |
| sin calibrar | 0,2268 | 0,6461 | 0,9636 | 6,8 | 17,2 |

**Lectura:** el calibrador baja un poco la validación contra la nota (−0,011 en AUC), pero **ninguna de las dos versiones pasa G8 ni G9**: en una pregunta de fortaleza la nota se concentra arriba y la señal es débil. G8/G9 miden contra un *proxy* (la nota); el calibrador se juzga contra etiquetas humanas (G11). Decisión: se mantiene el calibrador y **no se mueven los umbrales después de ver los resultados**. Pendiente para el equipo: más anotaciones de preguntas FORTALEZA (hoy 144, F1 CV 0,56).

In [ ]:
# 🧪 Anexo A.17 — G9: validación contra la nota, calibrado vs. sin calibrar (misma corrida)
dx_notas = encuestas[FOCAL].notas()
dx_con = resultado.clausulas
dx_sin = dx_con.copy()
dx_op = dx_sin["sentimiento"].notna()
dx_normal = dx_op & ~(dx_sin["motivo"] == "ALERTA")
dx_sin.loc[dx_normal, "polaridad"] = (dx_sin["p_pos"] - dx_sin["p_neg"])[dx_normal].round(4)
dx_ajustar = dx_normal & (dx_sin["sentimiento_modelo"] == "NEU") & (dx_sin["polaridad"].abs() >= cfg_final.margen_neu)
dx_sin.loc[dx_normal, "sentimiento"] = dx_sin.loc[dx_normal, "sentimiento_modelo"]
dx_sin.loc[dx_ajustar, "sentimiento"] = np.where(dx_sin.loc[dx_ajustar, "polaridad"] > 0, "POS", "NEG")
dx_filas = []
for nombre, marco in [("calibrado", dx_con), ("sin calibrar", dx_sin)]:
    v = ft.validar_contra_nota(marco, dx_notas, "ESTUDIANTE")
    dist = marco.loc[dx_op, "sentimiento"].value_counts(normalize=True)
    dx_filas.append({"versión": nombre, "spearman_doc (G8)": v["spearman_documento"],
                     "AUC (G9)": v["auc_nota_baja"], "monotonía": v["monotonia_deciles"],
                     "%NEG": round(100 * dist.get("NEG", 0), 1), "%NEU": round(100 * dist.get("NEU", 0), 1)})
print(pd.DataFrame(dx_filas).to_string(index=False))